# Two Dice: Analytical PMF and Monte Carlo Convergence

This notebook is the first probability artifact in `craps-lab`. It derives
the probability mass function for the sum of two fair six-sided dice from
first principles, visualizes it, and then validates the seeded `DiceRoller`
by running Monte Carlo simulations at increasing sample sizes and overlaying
the empirical frequencies on the closed form.

Every later chapter — house edges, come-bet expectations, shooter-length
distributions — starts from the numbers that fall out of the cells below.

## Derivation

Rolling two fair, independent six-sided dice gives a sample space of
$6 \times 6 = 36$ equally likely ordered pairs $(d_1, d_2)$.

For each target sum $s \in \{2, 3, \ldots, 12\}$, the probability is

$$
P(S = s) \;=\; \frac{\lvert \{(d_1, d_2) : d_1 + d_2 = s,\; 1 \le d_1, d_2 \le 6\} \rvert}{36}.
$$

Counting by hand gives the classical symmetric "tent" shape:

| $s$ | count | $P(S = s)$ |
|-----|:-----:|:-----:|
| 2  | 1 | 1/36 |
| 3  | 2 | 2/36 |
| 4  | 3 | 3/36 |
| 5  | 4 | 4/36 |
| 6  | 5 | 5/36 |
| 7  | 6 | 6/36 |
| 8  | 5 | 5/36 |
| 9  | 4 | 4/36 |
| 10 | 3 | 3/36 |
| 11 | 2 | 2/36 |
| 12 | 1 | 1/36 |

The most important number in craps is $P(S = 7) = 1/6$. It sets the
come-out natural probability, the 7-out probability during a point round,
and the long-run expected loss per roll on essentially every bet that
resolves against a seven.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

from craps_lab.dice import DiceRoller
from craps_lab.probability import (
    MAX_TWO_DICE_SUM,
    MIN_TWO_DICE_SUM,
    two_dice_sum_pmf,
)

plt.rcParams.update({
    "figure.figsize": (8, 4),
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

SEED = 0xC0FFEE
TOTALS = list(range(MIN_TWO_DICE_SUM, MAX_TWO_DICE_SUM + 1))

In [ ]:
pmf = two_dice_sum_pmf()
for total in TOTALS:
    frac = pmf[total]
    print(f"P(S = {total:2d}) = {str(frac):>5}  = {float(frac):.4f}")

In [ ]:
probabilities = [float(pmf[s]) for s in TOTALS]

fig, ax = plt.subplots()
ax.bar(TOTALS, probabilities, color="steelblue", edgecolor="white")
ax.set_xlabel("sum of two dice")
ax.set_ylabel("P(S = s)")
ax.set_title("Analytical PMF of 2d6 sum")
ax.set_xticks(TOTALS)
plt.tight_layout()
plt.show()

## Monte Carlo convergence

Now roll the seeded `DiceRoller` at four increasing sample sizes and overlay
the empirical frequencies on the analytical PMF. At $n = 10^3$ the empirical
bars rattle noticeably; at $n = 10^6$ they lie essentially on top of the
closed-form values.

In [ ]:
sample_sizes = [1_000, 10_000, 100_000, 1_000_000]

fig, axes = plt.subplots(1, len(sample_sizes), figsize=(16, 4), sharey=True)
for ax, n in zip(axes, sample_sizes, strict=True):
    roller = DiceRoller(seed=SEED)
    counts: Counter[int] = Counter(roller.roll().total for _ in range(n))
    empirical = [counts[s] / n for s in TOTALS]

    width = 0.4
    ax.bar(
        [s - width / 2 for s in TOTALS],
        probabilities,
        width=width,
        color="steelblue",
        edgecolor="white",
        label="analytical",
    )
    ax.bar(
        [s + width / 2 for s in TOTALS],
        empirical,
        width=width,
        color="tomato",
        edgecolor="white",
        label="empirical",
    )
    ax.set_xlabel("sum")
    ax.set_title(f"n = {n:,}")
    ax.set_xticks(TOTALS)

axes[0].set_ylabel("probability")
axes[0].legend(loc="upper left")
plt.tight_layout()
plt.show()

## Convergence rate: the running estimate of $P(S = 7)$

The running estimate $\hat{p}_n = \frac{1}{n}\sum_{i=1}^{n} \mathbf{1}[S_i = 7]$
is a consistent, unbiased estimator of the analytical $1/6$. Its standard
error is $\sqrt{p(1 - p)/n} \approx 0.373 / \sqrt{n}$, so the expected
$\pm 2\,\text{SEM}$ band shrinks as $n^{-1/2}$.

The plot below uses a log-$n$ axis so the early, noisy phase and the late,
tight phase are both visible in the same figure.

In [ ]:
max_n = 200_000
roller = DiceRoller(seed=SEED)

running = np.zeros(max_n)
seven_count = 0
for i in range(max_n):
    if roller.roll().total == 7:
        seven_count += 1
    running[i] = seven_count / (i + 1)

analytical = 1.0 / 6.0
n_axis = np.arange(1, max_n + 1)
sem_band = 2.0 * np.sqrt(analytical * (1.0 - analytical) / n_axis)

fig, ax = plt.subplots()
ax.plot(n_axis, running, color="tomato", linewidth=1, label="running empirical")
ax.axhline(analytical, color="steelblue", linestyle="--", label="analytical = 1/6")
ax.fill_between(
    n_axis,
    analytical - sem_band,
    analytical + sem_band,
    color="steelblue",
    alpha=0.15,
    label="±2 SEM",
)
ax.set_xscale("log")
ax.set_xlabel("rolls")
ax.set_ylabel("P(S = 7)")
ax.set_ylim(0.0, 0.3)
ax.set_title("Running estimate of P(S = 7) vs. number of rolls")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## Takeaways

1. **The PMF is exact.** `craps_lab.probability.two_dice_sum_pmf()` returns
   `Fraction` values, so every downstream analytic computation (house
   edges, expected loss per roll, variance decompositions) can stay in
   rational arithmetic until the final display step.
2. **The Monte Carlo converges cleanly.** At $n = 10^6$ the side-by-side
   bar charts are visually indistinguishable from the analytical PMF, and
   the running estimate of $P(S = 7)$ stabilizes on $1/6$ well inside the
   $\pm 2\,\text{SEM}$ band.
3. **Seeded determinism.** Because `DiceRoller` takes a seed, the whole
   notebook re-runs to identical figures — which is the foundation that
   lets the unit test in `tests/test_convergence.py` make a tight 5-sigma
   convergence assertion.

Next up: translating raw dice probabilities into *bet-level* probabilities
($P(\text{pass line wins})$, $P(\text{don't pass wins})$, the place-bet
table) and deriving the house edge for each one from first principles.